# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*

**Conestoga College | PROG8245 | Group 2**

Objective: build and verify Boolean, phrase, and ranked retrieval on a real corpus.

| Member | Student ID |
|---|---|
| [Name 1] | [ID 1] |
| [Name 2] | [ID 2] |
| [Name 3] | [ID 3] |


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (90 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Manual IR and Inverted Matrix coding + Markdown documentation (work as teams)
3. **Push to GitHub** *(15 min)* – Teams commit and push initial notebooks. **Make sure to include your names so it is easy to identify the team that developed the inverted index**.
4. **Instructor Review** - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**
5. **Email Delivery** *(15 min)* – Each team send the instructor an email **with the *.git link** to the GitHub repo **(one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. **Push to the course GitHub** *(automated)* – Find the last code cell in this notebook, update your team number, and run the cell. It will push your notebook to the repo at `submissions/team#`

## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and Inverted Index.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Example repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - Three meaningful commits by the actual team members on the existing Group 2 repository

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


The original workshop downloader saved each [Python Software Foundation RSS](https://pyfound.blogspot.com/feeds/posts/default?alt=rss) entry's title and HTML summary in numeric order. This run uses the existing local files and reads the 20 expected names only. It decodes entities and removes markup and hidden blocks. The snapshot date and a blanket redistribution license are unverified.

**Original team talking point (collection).** Markup can inflate vocabulary size with tag and attribute names. We count unique tokens from visible text before stop-word filtering and stemming.

In [1]:
from collections import Counter, defaultdict
from html.parser import HTMLParser
from pathlib import Path
import math
import re
import unicodedata

RSS_URL = 'https://pyfound.blogspot.com/feeds/posts/default?alt=rss'


class VisibleText(HTMLParser):
    block_tags = {'br', 'div', 'h1', 'h2', 'h3', 'h4', 'li', 'p', 'pre', 'section', 'td', 'tr'}
    hidden_tags = {'script', 'style', 'template'}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []
        self.hidden_depth = 0

    def handle_starttag(self, tag, attrs):
        if tag in self.hidden_tags:
            self.hidden_depth += 1
        elif not self.hidden_depth and tag in self.block_tags:
            self.parts.append(' ')

    def handle_endtag(self, tag):
        if tag in self.hidden_tags:
            self.hidden_depth = max(0, self.hidden_depth - 1)
        elif not self.hidden_depth and tag in self.block_tags:
            self.parts.append(' ')

    def handle_data(self, data):
        if not self.hidden_depth:
            self.parts.append(data)


def html_to_text(markup):
    parser = VisibleText()
    parser.feed(markup)
    parser.close()
    return ' '.join(''.join(parser.parts).split())


assert html_to_text('<p>Py<b>thon</b> &amp; IR</p><p>3</p><style>x</style>') == 'Python & IR 3'

In [2]:
def load_documents(folder):
    documents, titles = {}, {}
    for doc_id in range(20):
        path = folder / f'blog_{doc_id + 1}.txt'
        raw = path.read_text(encoding='utf-8')
        title, separator, body = raw.partition('\n\n')
        assert separator and title.strip() and body.strip(), f'Incomplete cache file: {path}'
        titles[doc_id] = html_to_text(title)
        documents[doc_id] = (path.name, html_to_text(raw))
    return documents, titles


documents, titles = load_documents(Path('sample_docs'))

In [3]:
TOKEN_PATTERN = re.compile(r"[^\W_]+(?:'[^\W_]+)*")


def tokenize(text):
    normalized = unicodedata.normalize('NFKC', text).casefold().replace(chr(8217), "'")
    return TOKEN_PATTERN.findall(normalized)


raw_tokens = {i: tokenize(text) for i, (_, text) in documents.items()}
vocabulary = {token for tokens in raw_tokens.values() for token in tokens}
print('Total documents:', len(documents))
print('Total words:', sum(map(len, raw_tokens.values())))
print('Unique words:', len(vocabulary))
assert len(documents) >= 20 and all(raw_tokens.values())
assert len({text for _, text in documents.values()}) == len(documents)
assert len(vocabulary) > 2000
print('Document collection requirements passed!')

Total documents: 20
Total words: 13309
Unique words: 2514
Document collection requirements passed!


## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


Unicode compatibility normalization and case folding standardize input. The pattern retains letters, digits and internal apostrophes in contractions. Underscores, hyphens and decimal points act as separators. The same tokenizer prepares corpus statistics and queries.

In [4]:
assert tokenize("PYTHON\tSoftware\ncan't won\u2019t 3.13 snake_case #IR!") == [
    'python', 'software', "can't", "won't", '3', '13', 'snake', 'case', 'ir'
]
assert tokenize("Stra\u00dfe \uff30ython co-op") == ['strasse', 'python', 'co', 'op']
assert tokenize("'   _ !") == []
print('Tokenizer boundary, case, punctuation, contraction and digit checks passed.')
print('Document 0 token preview:', raw_tokens[0][:12])

Tokenizer boundary, case, punctuation, contraction and digit checks passed.
Document 0 token preview: ['de', 'fanging', 'dangerous', 'triggers', 'in', 'github', 'actions', 'across', 'an', 'organization', 'authors', 'mike']


## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


Affix stripping removes a prefix or suffix. Porter stemming uses suffix rules and spelling adjustments, reducing *running* to *run* and *developers* to *develop*. Stems can include forms such as *softwar*. English stop words are removed before stemming.

**Original team talking point (linguistic processing).** Mapping *testing* and *tests* to *test* improves recall for those variants. Phrase queries retain common words such as *and* so they can enforce adjacency.

In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

cache = str(Path('.nltk_data').resolve())
nltk.data.path.insert(0, cache)
try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    nltk.data.path[:] = [cache]
    if not nltk.download('stopwords', download_dir=cache, quiet=True):
        raise RuntimeError('Install the NLTK English stopwords corpus before running.')
    stop_words = set(stopwords.words('english'))

stemmer = PorterStemmer()


def normalize_tokens(tokens):
    return [stemmer.stem(token) for token in tokens if token not in stop_words]


sample = tokenize('The RUNNING developers were testing software.')
filtered = [token for token in sample if token not in stop_words]
stems = normalize_tokens(sample)
assert filtered == ['running', 'developers', 'testing', 'software']
assert stems == ['run', 'develop', 'test', 'softwar']
print('Tokens:  ', sample)
print('Filtered:', filtered)
print('Stemmed: ', stems)

Tokens:   ['the', 'running', 'developers', 'were', 'testing', 'software']
Filtered: ['running', 'developers', 'testing', 'software']
Stemmed:  ['run', 'develop', 'test', 'softwar']


## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


**Original team talking point (Boolean retrieval).** A posting stores each document ID once, even for repeated terms. AND intersects postings. AND NOT subtracts the right posting from the left. A missing right term leaves the left posting intact for AND NOT. IDs are zero-based and resolve to the original filenames.

In [6]:
normalized_docs = {i: normalize_tokens(tokens) for i, tokens in raw_tokens.items()}
term_counts = {i: Counter(tokens) for i, tokens in normalized_docs.items()}


def build_inverted_index(counts):
    index = defaultdict(list)
    for doc_id in sorted(counts):
        for term in counts[doc_id]:
            index[term].append(doc_id)
    return dict(index)


inverted_index = build_inverted_index(term_counts)


def postings(query):
    terms = normalize_tokens(tokenize(query))
    if len(terms) > 1:
        raise ValueError('Use one searchable term per Boolean operand.')
    return set(inverted_index.get(terms[0], [])) if terms else set()


def boolean_and(left, right):
    return sorted(postings(left) & postings(right))


def boolean_and_not(left, right):
    return sorted(postings(left) - postings(right))


def show_docs(ids):
    return [(i, documents[i][0]) for i in ids]


for left, right in [('SOFTWARE', 'grants'), ('python', 'zzzxqabsent'),
                    ('zzzxqabsent', 'python'), ('the', 'python')]:
    left_terms = normalize_tokens(tokenize(left))
    right_terms = normalize_tokens(tokenize(right))
    present_left = [i for i, ts in normalized_docs.items() if left_terms and left_terms[0] in ts]
    present_right = {i for i, ts in normalized_docs.items() if right_terms and right_terms[0] in ts}
    intersection = boolean_and(left, right)
    difference = boolean_and_not(left, right)
    assert intersection == [i for i in present_left if i in present_right]
    assert difference == [i for i in present_left if i not in present_right]
    print(f'{left} AND {right}: {show_docs(intersection)}')
    print(f'{left} AND NOT {right}: {show_docs(difference)}')

assert boolean_and('software', 'grants') and boolean_and_not('software', 'grants')
assert all(ids == sorted(set(ids)) for ids in inverted_index.values())
print('Sample software posting:', inverted_index['softwar'])
print('Boolean results matched direct corpus scans.')

SOFTWARE AND grants: [(0, 'blog_1.txt'), (1, 'blog_2.txt'), (3, 'blog_4.txt'), (4, 'blog_5.txt'), (7, 'blog_8.txt'), (12, 'blog_13.txt'), (15, 'blog_16.txt'), (18, 'blog_19.txt')]
SOFTWARE AND NOT grants: [(2, 'blog_3.txt'), (5, 'blog_6.txt'), (8, 'blog_9.txt'), (10, 'blog_11.txt'), (13, 'blog_14.txt')]
python AND zzzxqabsent: []
python AND NOT zzzxqabsent: [(0, 'blog_1.txt'), (1, 'blog_2.txt'), (2, 'blog_3.txt'), (3, 'blog_4.txt'), (4, 'blog_5.txt'), (5, 'blog_6.txt'), (6, 'blog_7.txt'), (7, 'blog_8.txt'), (8, 'blog_9.txt'), (9, 'blog_10.txt'), (10, 'blog_11.txt'), (11, 'blog_12.txt'), (12, 'blog_13.txt'), (13, 'blog_14.txt'), (14, 'blog_15.txt'), (15, 'blog_16.txt'), (16, 'blog_17.txt'), (17, 'blog_18.txt'), (18, 'blog_19.txt'), (19, 'blog_20.txt')]
zzzxqabsent AND python: []
zzzxqabsent AND NOT python: []
the AND python: []
the AND NOT python: []
Sample software posting: [0, 1, 2, 3, 4, 5, 7, 8, 10, 12, 13, 15, 18]
Boolean results matched direct corpus scans.


## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries.
- Demonstrate that they return the correct documents.


Both phrase indexes use casefolded tokens with stop words and word endings intact. Positions start at zero in the combined title and summary. Exactness is defined over tokens, so matches can cross punctuation.

**Original team talking point (phrase search).** A biword posting directly answers two-token queries. For longer phrases, document-level biword intersections can combine pairs from different locations. The positional index shifts the position lists by the respective query offsets and intersects possible starts. It stores one offset per occurrence but avoids scanning the full text. We prefer a positional index when both bigrams and longer phrases are needed.

In [7]:
def build_positional_index(token_streams):
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in token_streams.items():
        for position, token in enumerate(tokens):
            index[token][doc_id].append(position)
    return index


def build_biword_index(token_streams):
    index = defaultdict(set)
    for doc_id, tokens in token_streams.items():
        for pair in zip(tokens, tokens[1:]):
            index[pair].add(doc_id)
    return {pair: sorted(ids) for pair, ids in index.items()}


def phrase_query(query, index):
    terms = tokenize(query)
    if not terms:
        return {}
    term_postings = [index.get(term, {}) for term in terms]
    candidates = set.intersection(*(set(posting) for posting in term_postings))
    matches = {}
    for doc_id in sorted(candidates):
        starts = set(term_postings[0][doc_id])
        for offset, posting in enumerate(term_postings[1:], start=1):
            starts.intersection_update(pos - offset for pos in posting[doc_id])
        if starts:
            matches[doc_id] = sorted(starts)
    return matches


positional_index = build_positional_index(raw_tokens)
biword_index = build_biword_index(raw_tokens)

## 🧠 Additional Challenge: Implement Positional Indexes

Implement Positional Indexes, an advanced version of an inverted index that not only stores which documents a term appears in, but also where (at what positions) the term occurs within each document.

Apply it to the documents you used to produce the Inverted Matrix during the Active Learning Workshop. 

Then compare it against a **Term Document Count Matrix** (which you don't have to implement)

And finally, fill in the blanks in the table below.

| Term | What is it? | How is it used? | Pros | Cons | 
|----------------|-------------|-------------|-------------|-------------|
| Term Frequency (TF)  |  |  |  |
| Inverse Document Frequency (idF weight)  |  |  |  |

Then use the table to prepare talking points on:
- Which implementation you prefer to use for searching bigrams (a.k.a., biwords), pairs of consecutive words in a document, and why?

#### Sample solution:

In [8]:
def scan_phrase(query):
    phrase = tokenize(query)
    hits = {}
    for doc_id, tokens in raw_tokens.items():
        starts = [i for i in range(len(tokens) - len(phrase) + 1)
                  if tokens[i:i + len(phrase)] == phrase]
        if starts:
            hits[doc_id] = starts
    return hits


for query in ['python software', 'security developer', 'foundation software python']:
    hits = phrase_query(query, positional_index)
    assert hits == scan_phrase(query)
    print(f'{query!r}: {len(hits)} documents, {sum(map(len, hits.values()))} matches')
    print('  Files:', show_docs(hits))
    print('  Example offsets:', list(hits.items())[:2])

assert phrase_query('python software', positional_index)
assert phrase_query('security developer', positional_index)
assert not phrase_query('foundation software python', positional_index)
pair = ('python', 'software')
assert biword_index[pair] == list(scan_phrase(' '.join(pair)))
print('Biword postings:', show_docs(biword_index[pair]))

regression_index = build_positional_index({0: tokenize('Python and software software')})
assert phrase_query('python software', regression_index) == {}
assert phrase_query('SOFTWARE software', regression_index) == {0: [2]}
assert phrase_query('!', regression_index) == {}
print('Phrase offsets passed scan and adjacency checks.')

'python software': 11 documents, 19 matches
  Files: [(0, 'blog_1.txt'), (1, 'blog_2.txt'), (2, 'blog_3.txt'), (3, 'blog_4.txt'), (4, 'blog_5.txt'), (5, 'blog_6.txt'), (7, 'blog_8.txt'), (8, 'blog_9.txt'), (10, 'blog_11.txt'), (12, 'blog_13.txt'), (13, 'blog_14.txt')]
  Example offsets: [(0, [151, 791]), (1, [14])]
'security developer': 4 documents, 9 matches
  Files: [(0, 'blog_1.txt'), (1, 'blog_2.txt'), (2, 'blog_3.txt'), (4, 'blog_5.txt')]
  Example offsets: [(0, [137, 693]), (1, [963])]
'foundation software python': 0 documents, 0 matches
  Files: []
  Example offsets: []
Biword postings: [(0, 'blog_1.txt'), (1, 'blog_2.txt'), (2, 'blog_3.txt'), (3, 'blog_4.txt'), (4, 'blog_5.txt'), (5, 'blog_6.txt'), (7, 'blog_8.txt'), (8, 'blog_9.txt'), (10, 'blog_11.txt'), (12, 'blog_13.txt'), (13, 'blog_14.txt')]
Phrase offsets passed scan and adjacency checks.


### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

### 📝 Completed Table: Term Frequency (TF) and Inverse Document Frequency (IDF weight)

| Term                           | What is it?                                                                 | How is it used?                          | Pros                                   | Cons                                   |
|-------------------------------|----------------------------------------------------------------------------|------------------------------------------|----------------------------------------|----------------------------------------|
| Term Frequency (TF)            | Number of times a term appears in a document                               | Measures term importance in a document   | Simple, fast, good for ranking         | Doesn't account for common terms       |
| Inverse Document Frequency     | Log ratio of total docs to docs containing the term                        | Weights rare terms higher                | Reduces impact of common words         | Can overweight rare, irrelevant terms  |
| (IDF weight)                   | Used with TF to score term relevance: TF-IDF = TF * IDF                    | Improves search relevance                | Highlights unique, important terms     | Requires global corpus statistics      |

Let's implement the Inverse Document Frequency (IDF)

In [9]:
doc_frequency = {term: len(ids) for term, ids in inverted_index.items()}
idf_scores = {term: math.log(len(documents) / df) for term, df in doc_frequency.items()}
term = normalize_tokens(tokenize('software'))[0]
df = sum(term in tokens for tokens in normalized_docs.values())
assert doc_frequency[term] == df
assert math.isclose(idf_scores[term], math.log(len(documents) / df))
print(f'{term}: N={len(documents)}, DF={df}, IDF={idf_scores[term]:.6f}')

softwar: N=20, DF=13, IDF=0.430783


### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

In [10]:
tf = term_counts[0][term]
assert tf == normalized_docs[0].count(term)
print(f'{term} in document 0: TF={tf}, TF-IDF={tf * idf_scores[term]:.6f}')
print('TF by document:', {i: counts[term] for i, counts in term_counts.items()})

softwar in document 0: TF=2, TF-IDF=0.861566
TF by document: {0: 2, 1: 1, 2: 6, 3: 3, 4: 6, 5: 3, 6: 0, 7: 1, 8: 1, 9: 0, 10: 1, 11: 0, 12: 1, 13: 1, 14: 0, 15: 1, 16: 0, 17: 0, 18: 2, 19: 0}


#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Substitute the actual values from your output to see the final TF-IDF score for Document 0.

In [11]:
def query_processor(query, top_k=5):
    scores = defaultdict(float)
    for term in sorted(set(normalize_tokens(tokenize(query)))):
        for doc_id in inverted_index.get(term, ()):
            scores[doc_id] += term_counts[doc_id][term] * idf_scores[term]
    ranked = sorted(((i, score) for i, score in scores.items() if score > 0),
                    key=lambda result: (-result[1], result[0]))
    return ranked[:top_k]


ranking_query = 'Python software foundation grants'
ranking = query_processor(ranking_query)
assert ranking and query_processor('zzzxqabsent') == []
query_terms = set(normalize_tokens(tokenize(ranking_query)))
reference = []
for doc_id, tokens in normalized_docs.items():
    weight = sum(tokens.count(t) * math.log(len(documents) / doc_frequency[t])
                 for t in sorted(query_terms & doc_frequency.keys()))
    if weight > 0:
        reference.append((doc_id, weight))
reference.sort(key=lambda item: (-item[1], item[0]))
assert [i for i, _ in ranking] == [i for i, _ in reference[:5]]
assert all(math.isclose(actual, expected) for (_, actual), (_, expected) in zip(ranking, reference))
for t in sorted(query_terms):
    print(f'{t:10s} DF={doc_frequency.get(t, 0):2d} IDF={idf_scores.get(t, 0):.6f}')
for doc_id, score in ranking:
    print(f'{score:8.4f}  [{doc_id}] {documents[doc_id][0]}  {titles[doc_id]}')
print('Ranking matched direct TF-IDF calculations.')

foundat    DF=11 IDF=0.597837
grant      DF= 9 IDF=0.798508
python     DF=20 IDF=0.000000
softwar    DF=13 IDF=0.430783
  5.1767  [4] blog_5.txt  Announcing new Security Developer at the PSF
  4.3782  [2] blog_3.txt  How the Python ecosystem responded to the eMudhra intermediate compromise
  4.2227  [1] blog_2.txt  Python is for everyone: Join in the PSF year-end fundraiser & membership drive!
  4.0556  [18] blog_19.txt  Nina Zakharenko: 2026 PSF Board Election Candidate Interview
  3.8844  [3] blog_4.txt  Joshua Mcshannon Joins the PSF as Infrastructure Engineer
Ranking matched direct TF-IDF calculations.


### Ranked retrieval for RAG and an AI agent

The ranker sums raw TF times natural-log IDF across distinct query terms. More mentions increase a term's contribution when IDF is positive. A term found in every document contributes zero. Longer documents may accumulate more matches under this score.

**Original team talking point (ranking and applications).** A rare term can dominate the score even if it is incidental to the question. Excerpts help readers check whether the evidence actually answers it.

RAG passes retrieved excerpts and IDs into a model prompt for a grounded response with citations. This local walkthrough constructs the prompt. A simple evidence-seeking agent calls retrieval, checks source IDs and excerpt provenance, and reports insufficient evidence when retrieval is empty. With a model, the next step would be to generate an answer and verify each claim against its cited excerpt.

In [12]:
def retrieve_sources(question, top_k=2):
    terms = set(normalize_tokens(tokenize(question)))
    sources = []
    for doc_id, score in query_processor(question, top_k):
        filename, text = documents[doc_id]
        words = text.split()
        anchors = ((idf_scores[t], -i) for i, word in enumerate(words)
                   for t in normalize_tokens(tokenize(word)) if t in terms)
        _, offset = max(anchors, default=(0, 0))
        start = -offset
        excerpt = ' '.join(words[max(0, start - 4):start + 12])
        sources.append({'id': doc_id, 'file': filename, 'score': score, 'excerpt': excerpt})
    return sources


def answer_with_evidence(question):
    sources = retrieve_sources(question)
    if not sources:
        return {'action': 'retrieve_sources', 'status': 'insufficient evidence'}
    context = '\n'.join(f"[{s['id']}] {s['file']}: {s['excerpt']}" for s in sources)
    prompt = (
        'Use the excerpts as evidence. Cite supporting document IDs in brackets. '
        'If the excerpts do not answer the question, say so.\n\n'
        f'Question: {question}\n\nExcerpts:\n{context}'
    )
    return {'action': 'retrieve_sources', 'status': 'prompt ready',
            'sources': sources, 'prompt': prompt}


result = answer_with_evidence('Who is the new security developer?')
assert result['status'] == 'prompt ready'
for source in result['sources']:
    filename, text = documents[source['id']]
    assert source['file'] == filename and source['excerpt'] in text
    assert f"[{source['id']}]" in result['prompt']
assert answer_with_evidence('zzzxqabsent')['status'] == 'insufficient evidence'
print('Agent:', result['action'], '->', result['status'])
print(result['prompt'])

Agent: retrieve_sources -> prompt ready
Use the excerpts as evidence. Cite supporting document IDs in brackets. If the excerpts do not answer the question, say so.

Question: Who is the new security developer?

Excerpts:
[4] blog_5.txt: Announcing new Security Developer at the PSF We’re excited to announce that Brett Smith
[1] blog_2.txt: and PyPI progress Increase security across the Python ecosystem Bring the global Python community together Make


## 🧠 Reflection

Boolean retrieval combines document membership. Positional retrieval can require order
and adjacency. A ranking prioritizes the strongest combination of query term frequency
and rarity. The reader still checks the source before making a factual claim.

**Original team talking point (reproducibility).** Stable file order and document-ID tie
breaks make repeated runs comparable. Scan checks let teammates verify results during review.

**Team checklist.** Confirm three names and IDs, review saved outputs, and record three
genuine team commits. Record peer feedback after review. Complete the Group 2 PDF and
coordinate the instructor email and Dropbox submission.

## 🧠 Push your code to the course GitHub repository

Update your team number and then run the code. 

    !python submit_assignment.py --notebook IR_InvertedMatrix_Workshop.ipynb --student_id "teamA"

Run All executes the workshop locally. Obtain the official script and confirm the Group 2 identifier before using the submission command.